In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.lines import Line2D

import games

FPS = games.GAMES[games.GAME]['fps']
N_FRAMES = games.GAMES[games.GAME]['n_video_frames']
PUBLIC_FIRST, PUBLIC_LAST = 11185, 18684  # first 5 min of play — see DATA.md
KICKOFF_1ST, KICKOFF_2ND = 11185, 113742

GT_PATH = os.path.join(
    os.path.abspath('../..'), 'evaluation', 'gt', 'BAR-ATH_possession_gt_public.csv')

print('game:        ', games.GAME)
print('fps:         ', FPS)
print('public GT:   ', GT_PATH)
print('public span: ', PUBLIC_FIRST, '…', PUBLIC_LAST,
      '(%d frames / %.1f s)' % (PUBLIC_LAST - PUBLIC_FIRST + 1,
                               (PUBLIC_LAST - PUBLIC_FIRST + 1) / FPS))

In [ ]:
# Published counts from DATA.md — the XML is not required to know the vocabulary.
SET_PIECE_TYPES = pd.DataFrame([
    ('Throw-in',              26, 'N/A on every throw-in'),
    ('Corner Kick',           12, 'Team tagged'),
    ('Goal Kick',             10, 'Team tagged'),
    ('Free Kick',              5, 'Team tagged'),
    ('Kick Off (After Goal)',  4, 'Team tagged'),
    ('Kick Off (Start)',       2, 'contains real kickoff frames'),
], columns=['Type', 'n_full_match', 'team_note'])

SET_PIECE_TYPES['share_of_set_pieces'] = (
    SET_PIECE_TYPES['n_full_match'] / SET_PIECE_TYPES['n_full_match'].sum())

display(SET_PIECE_TYPES)

fig, ax = plt.subplots(figsize=(8, 3.2))
colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#B279A2', '#FF9DA6']
ax.barh(SET_PIECE_TYPES['Type'], SET_PIECE_TYPES['n_full_match'], color=colors)
ax.invert_yaxis()
ax.set_xlabel('tagged intervals (full match, from DATA.md)')
ax.set_title('Smart Tagging SET PIECES vocabulary — 59 intervals')
for i, n in enumerate(SET_PIECE_TYPES['n_full_match']):
    ax.text(n + 0.3, i, str(n), va='center', fontsize=9)
ax.set_xlim(0, SET_PIECE_TYPES['n_full_match'].max() + 4)
plt.tight_layout()
plt.show()

In [ ]:
# Approximate reconstruction from starter/01_smart_tagging.ipynb notes.
# Shape: opens ~3.5 s before the strike, lasts ~5 s, contains the real kickoff.
EARLY_S, WINDOW_S = 3.5, 5.0

def approx_setpiece_window(strike_frame, early_s=EARLY_S, window_s=WINDOW_S):
    start = int(round(strike_frame - early_s * FPS))
    end = start + int(round(window_s * FPS))
    return start, end

documented_kickoffs = pd.DataFrame([
    {
        'Type': 'Kick Off (Start)',
        'Half': '1st Half',
        'real_kickoff': KICKOFF_1ST,
        'start_frame': approx_setpiece_window(KICKOFF_1ST)[0],
        'end_frame': approx_setpiece_window(KICKOFF_1ST)[1],
        'source': 'documented shape around published kickoff',
        'in_public_gt': True,
    },
    {
        'Type': 'Kick Off (Start)',
        'Half': '2nd Half',
        'real_kickoff': KICKOFF_2ND,
        'start_frame': approx_setpiece_window(KICKOFF_2ND)[0],
        'end_frame': approx_setpiece_window(KICKOFF_2ND)[1],
        'source': 'documented shape around published kickoff',
        'in_public_gt': False,  # public GT is X here
    },
])
documented_kickoffs['contains_kickoff'] = (
    (documented_kickoffs['start_frame'] <= documented_kickoffs['real_kickoff'])
    & (documented_kickoffs['real_kickoff'] <= documented_kickoffs['end_frame'])
)
display(documented_kickoffs)

In [ ]:
gt = pd.read_csv(GT_PATH)
# Public file is full-length (199,319 rows) with real labels only on frames 11185…18684.
print('gt rows:', len(gt), '  expected ~', N_FRAMES)

public = gt[(gt['frame'] >= PUBLIC_FIRST) & (gt['frame'] <= PUBLIC_LAST)].copy()
print(public['label'].value_counts().reindex(['H', 'A', 'D', 'X']).fillna(0).astype(int))
print()
print('non-X frames: %d (expect 7500)' % (public['label'] != 'X').sum())


def label_spells(frame_label_df):
    """Contiguous constant-label runs → DataFrame with start/end inclusive."""
    frames = frame_label_df['frame'].to_numpy()
    labs = frame_label_df['label'].to_numpy()
    change = np.flatnonzero(labs[1:] != labs[:-1]) + 1
    starts = np.r_[0, change]
    ends = np.r_[change, len(labs)]
    rows = []
    for s, e in zip(starts, ends):
        rows.append({
            'start_frame': int(frames[s]),
            'end_frame': int(frames[e - 1]),
            'n_frames': int(e - s),
            'duration_s': (e - s) / FPS,
            'label': labs[s],
        })
    return pd.DataFrame(rows)


spells = label_spells(public)
dead = spells[spells['label'] == 'D'].reset_index(drop=True)
alive = spells[spells['label'].isin(['H', 'A'])].reset_index(drop=True)

# A restart, for our purposes: the first alive frame after a dead-ball stretch.
restarts = []
for i, row in dead.iterrows():
    nxt = spells[spells['start_frame'] == row['end_frame'] + 1]
    if len(nxt) == 1 and nxt.iloc[0]['label'] in ('H', 'A'):
        restarts.append({
            'dead_start': row['start_frame'],
            'dead_end': row['end_frame'],
            'dead_s': row['duration_s'],
            'restart_frame': int(nxt.iloc[0]['start_frame']),
            'restart_to': nxt.iloc[0]['label'],
            'seconds_from_kickoff': (nxt.iloc[0]['start_frame'] - KICKOFF_1ST) / FPS,
        })
restarts = pd.DataFrame(restarts)

print('\npossession / dead-ball spells in public window:')
display(spells)
print('dead-ball stretches → next possession (GT restarts):')
display(restarts)

In [ ]:
COLOR = {'H': '#A50044', 'A': '#EE2523', 'D': '#1B4F72', 'X': '#BDBDBD'}
# Athletic uses a lighter red here so H/A stay distinguishable on one lane.
COLOR['A'] = '#F4A261'
SETPIECE_COLOR = {
    'Kick Off (Start)': '#2A9D8F',
    'Kick Off (After Goal)': '#264653',
    'Throw-in': '#E9C46A',
    'Corner Kick': '#E76F51',
    'Goal Kick': '#8AB17D',
    'Free Kick': '#9B5DE5',
}


def plot_gt_deadball_timeline(spells_df, restarts_df, kickoff_df,
                              frame_lo=PUBLIC_FIRST, frame_hi=PUBLIC_LAST,
                              title=None):
    fig, axes = plt.subplots(
        2, 1, figsize=(14, 3.6), sharex=True,
        gridspec_kw={'height_ratios': [1.0, 2.2], 'hspace': 0.08})
    ax_tag, ax_gt = axes

    # --- Smart Tag lane (documented kickoffs only) ---
    for _, row in kickoff_df.iterrows():
        if row['end_frame'] < frame_lo or row['start_frame'] > frame_hi:
            continue
        s = max(row['start_frame'], frame_lo)
        e = min(row['end_frame'], frame_hi)
        ax_tag.barh(0, e - s + 1, left=s, height=0.7, align='center',
                    color=SETPIECE_COLOR[row['Type']], edgecolor='black', lw=0.4)
        ax_tag.axvline(row['real_kickoff'], color='black', ls=':', lw=1.0, alpha=0.8)

    ax_tag.set_yticks([0])
    ax_tag.set_yticklabels(['SET PIECES\n(documented)'])
    ax_tag.set_ylim(-0.7, 0.7)
    ax_tag.set_title(title or 'Public GT — dead ball vs documented Kick Off (Start)')
    ax_tag.tick_params(axis='x', labelbottom=False)

    # --- GT possession lane ---
    for _, row in spells_df.iterrows():
        if row['label'] == 'X':
            continue
        ax_gt.barh(0, row['n_frames'], left=row['start_frame'], height=0.85,
                   align='center', color=COLOR[row['label']], linewidth=0)

    for _, row in restarts_df.iterrows():
        ax_gt.axvline(row['restart_frame'], color='black', ls='--', lw=1.1, alpha=0.85)
        ax_gt.annotate(
            'restart→%s' % row['restart_to'],
            xy=(row['restart_frame'], 0.45),
            xytext=(5, 8), textcoords='offset points',
            fontsize=8, rotation=90, va='bottom')

    ax_gt.set_yticks([0])
    ax_gt.set_yticklabels(['GT label'])
    ax_gt.set_ylim(-0.7, 0.7)
    ax_gt.set_xlim(frame_lo, frame_hi)
    ax_gt.set_xlabel('video frame  (seconds from 1st-half kickoff on top axis)')

    ax_top = ax_tag.twiny()
    ax_top.set_xlim((frame_lo - KICKOFF_1ST) / FPS, (frame_hi - KICKOFF_1ST) / FPS)
    ax_top.set_xlabel('seconds from 1st-half kickoff')

    legend = [
        Patch(facecolor=COLOR['H'], label='H (home possession)'),
        Patch(facecolor=COLOR['A'], label='A (away possession)'),
        Patch(facecolor=COLOR['D'], label='D (dead ball)'),
        Patch(facecolor=SETPIECE_COLOR['Kick Off (Start)'],
              edgecolor='black', label='Kick Off (Start) ≈window'),
        Line2D([0], [0], color='black', ls='--', label='GT restart (D→alive)'),
        Line2D([0], [0], color='black', ls=':', label='real kickoff frame'),
    ]
    ax_gt.legend(handles=legend, loc='upper right', ncol=3, fontsize=8, framealpha=0.92)
    return fig, axes


fig, _ = plot_gt_deadball_timeline(spells, restarts, documented_kickoffs)
plt.show()

In [ ]:
PAD_S = 8  # seconds of context on each side
pad = int(PAD_S * FPS)

n = len(restarts)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.1 * n), sharex=False)
if n == 1:
    axes = [axes]

for ax, (_, r) in zip(axes, restarts.iterrows()):
    lo = max(PUBLIC_FIRST, r['dead_start'] - pad)
    hi = min(PUBLIC_LAST, r['restart_frame'] + pad)
    window_spells = spells[
        (spells['end_frame'] >= lo) & (spells['start_frame'] <= hi)
    ]
    for _, sp in window_spells.iterrows():
        s = max(sp['start_frame'], lo)
        e = min(sp['end_frame'], hi)
        ax.barh(0, e - s + 1, left=s, height=0.8, align='center',
                color=COLOR[sp['label']], linewidth=0)

    # Highlight the dead stretch edge.
    ax.axvspan(r['dead_start'], r['dead_end'], color=COLOR['D'], alpha=0.15)
    ax.axvline(r['restart_frame'], color='black', ls='--', lw=1.2)

    # Documented kickoff window if it overlaps this zoom.
    for _, ko in documented_kickoffs.iterrows():
        if ko['end_frame'] < lo or ko['start_frame'] > hi:
            continue
        ax.barh(0.95, ko['end_frame'] - ko['start_frame'] + 1, left=ko['start_frame'],
                height=0.25, align='center',
                color=SETPIECE_COLOR[ko['Type']], edgecolor='black', lw=0.4)

    ax.set_xlim(lo, hi)
    ax.set_ylim(-0.6, 1.2)
    ax.set_yticks([])
    ax.set_title(
        'D %d-%d (%.1f s)  →  restart frame %d (%s)   t=+%.1fs from kickoff'
        % (r['dead_start'], r['dead_end'], r['dead_s'],
           r['restart_frame'], r['restart_to'], r['seconds_from_kickoff']),
        fontsize=10, loc='left')

    ax2 = ax.twiny()
    ax2.set_xlim((lo - KICKOFF_1ST) / FPS, (hi - KICKOFF_1ST) / FPS)
    ax2.set_xlabel('s from kickoff')

axes[-1].set_xlabel('video frame')
fig.legend(handles=[
    Patch(facecolor=COLOR['H'], label='H'),
    Patch(facecolor=COLOR['A'], label='A'),
    Patch(facecolor=COLOR['D'], label='D'),
    Line2D([0], [0], color='black', ls='--', label='restart'),
], loc='upper right', ncol=4, fontsize=8)
fig.suptitle('Dead-ball zooms (public GT) — SET PIECES should sit near each restart',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
n_public = PUBLIC_LAST - PUBLIC_FIRST + 1
n_dead = int((public['label'] == 'D').sum())
n_alive = int(public['label'].isin(['H', 'A']).sum())

summary = pd.DataFrame([
    ('public window frames', n_public),
    ('alive (H+A) frames', n_alive),
    ('dead (D) frames', n_dead),
    ('dead share', '%.1f%%' % (100 * n_dead / n_public)),
    ('dead-ball stretches', len(dead)),
    ('GT restarts (D→alive)', len(restarts)),
    ('mean dead stretch', '%.1f s' % dead['duration_s'].mean()),
    ('documented SET PIECES overlapping public window',
     int(((documented_kickoffs['start_frame'] <= PUBLIC_LAST)
          & (documented_kickoffs['end_frame'] >= PUBLIC_FIRST)).sum())),
    ('full-match SET PIECES (DATA.md)', 59),
], columns=['metric', 'value'])
display(summary)

# How much of each dead stretch would a ±W-second set-piece window around the restart cover?
windows_s = [2, 3.5, 5, 8]
rows = []
for _, r in restarts.iterrows():
    dead_frames = set(range(r['dead_start'], r['dead_end'] + 1))
    for w in windows_s:
        half = int(round(w * FPS / 2))
        # Centre the hypothetical tag on the restart (end of D), matching "window around restart".
        tag_lo = r['restart_frame'] - half
        tag_hi = r['restart_frame'] + half
        tag = set(range(tag_lo, tag_hi + 1))
        overlap = len(dead_frames & tag)
        rows.append({
            'restart_frame': r['restart_frame'],
            'dead_s': round(r['dead_s'], 1),
            'window_s': w,
            'overlap_frames': overlap,
            'overlap_share_of_D': overlap / len(dead_frames),
            'tag_covers_restart': tag_lo <= r['restart_frame'] <= tag_hi,
        })
coverage = pd.DataFrame(rows)
pivot = coverage.pivot(index='restart_frame', columns='window_s',
                       values='overlap_share_of_D')
print('\nShare of each GT dead stretch overlapped by a centred ±window/2 tag around the restart:')
display(pivot.map(lambda x: '%.1f%%' % (100 * x)))

fig, ax = plt.subplots(figsize=(8, 3.4))
x = np.arange(len(restarts))
width = 0.18
for i, w in enumerate(windows_s):
    vals = coverage[coverage['window_s'] == w]['overlap_share_of_D'].to_numpy()
    ax.bar(x + (i - 1.5) * width, vals, width, label='±%.1fs window' % (w / 2))
ax.set_xticks(x)
ax.set_xticklabels(['restart %d' % f for f in restarts['restart_frame']], rotation=15)
ax.set_ylabel('fraction of that D stretch covered')
ax.set_ylim(0, 1.05)
ax.set_title('Hypothetical SET PIECES window vs GT dead ball\n'
             '(tags mark the restart — they do not span the whole stoppage)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(
    'Takeaway: a ~5 s set-piece tag covers the restart and only the tail of a long dead-ball '
    'stretch. Short stoppages (≈5 s) can be almost fully covered; 15–20 s stoppages are mostly '
    'outside the tag. Smart Tagging is a restart prior, not a dead-ball segmenter.'
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))

# Schematic: long D bar, short SET PIECES window at the right edge, then alive.
ax.add_patch(FancyBboxPatch((0, 0.35), 8.0, 0.3, boxstyle='round,pad=0.02',
                            facecolor=COLOR['D'], edgecolor='none'))
ax.add_patch(FancyBboxPatch((7.2, 0.28), 1.2, 0.44, boxstyle='round,pad=0.02',
                            facecolor=SETPIECE_COLOR['Throw-in'],
                            edgecolor='black', linewidth=1.0, alpha=0.9))
ax.add_patch(FancyBboxPatch((8.0, 0.35), 3.5, 0.3, boxstyle='round,pad=0.02',
                            facecolor=COLOR['H'], edgecolor='none'))
ax.axvline(8.0, color='black', ls='--', lw=1.2)
ax.text(4.0, 0.85, 'GT dead ball (D) — can be long', ha='center', fontsize=10)
ax.text(7.8, 0.05, 'SET PIECES\n(~5 s window)', ha='center', fontsize=9,
        color='#333333')
ax.text(9.7, 0.85, 'alive (H/A)', ha='center', fontsize=10)
ax.text(8.05, 1.15, 'restart', ha='left', fontsize=9)
ax.set_xlim(-0.3, 12)
ax.set_ylim(0, 1.4)
ax.axis('off')
ax.set_title('How Smart Tagging covers dead ball: localise the restart, not the stoppage',
             loc='left')

# Type prior strip.
ax2 = fig.add_axes([0.1, 0.08, 0.8, 0.18])
left = 0
for (_, row), c in zip(SET_PIECE_TYPES.iterrows(), colors):
    ax2.barh(0, row['n_full_match'], left=left, color=c, edgecolor='white', height=0.6)
    if row['n_full_match'] >= 5:
        ax2.text(left + row['n_full_match'] / 2, 0, row['Type'].replace(' ', '\n'),
                 ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    left += row['n_full_match']
ax2.set_xlim(0, SET_PIECE_TYPES['n_full_match'].sum())
ax2.set_yticks([])
ax2.set_xlabel('full-match SET PIECES count (DATA.md) — Throw-ins dominate, Team=N/A on all of them')
plt.show()

In [ ]:
# --- GT restart × Smart Tagging coverage audit (no XML) -----------------------------
#
# Coverage definition (from DATA.md / load_smart_tagging.py):
#   a SET PIECES tag covers a restart iff start_frame <= restart_frame <= end_frame.
# Confirmed facts without the file:
#   - both Kick Off (Start) tags contain the real kickoffs
#   - 59 SET PIECES exist for the full match; types/counts in SET_PIECE_TYPES
#   - tags are ~5 s windows around the restart, not the whole dead-ball stretch

def gt_restarts_public(restarts_df, kickoff_frame=KICKOFF_1ST):
    """All restart events visible in the public GT window.

    Includes the opening kickoff (half starts alive — no preceding public D) and every
    D→alive transition already computed above.
    """
    rows = [{
        'restart_id': 'KO1',
        'restart_frame': int(kickoff_frame),
        'kind': 'kickoff_start',
        'dead_start': None,
        'dead_end': None,
        'dead_s': None,
        'restart_to': public.loc[public['frame'] == kickoff_frame, 'label'].iloc[0],
        'seconds_from_kickoff': 0.0,
    }]
    for i, r in restarts_df.reset_index(drop=True).iterrows():
        rows.append({
            'restart_id': 'R%d' % (i + 1),
            'restart_frame': int(r['restart_frame']),
            'kind': 'dead_to_alive',
            'dead_start': int(r['dead_start']),
            'dead_end': int(r['dead_end']),
            'dead_s': float(r['dead_s']),
            'restart_to': r['restart_to'],
            'seconds_from_kickoff': float(r['seconds_from_kickoff']),
        })
    return pd.DataFrame(rows)


def documented_tag_hit(restart_frame, kickoff_tags):
    """Return the documented Kick Off (Start) row that contains restart_frame, or None."""
    for _, tag in kickoff_tags.iterrows():
        if tag['start_frame'] <= restart_frame <= tag['end_frame']:
            return tag
    return None


def audit_restart_coverage(gt_restart_df, kickoff_tags, set_piece_types):
    """Per-restart coverage status using only documented Smart Tagging facts."""
    n_set_pieces_full = int(set_piece_types['n_full_match'].sum())
    out = []
    for _, r in gt_restart_df.iterrows():
        hit = documented_tag_hit(r['restart_frame'], kickoff_tags)
        if hit is not None:
            status = 'confirmed'
            evidence = (
                'documented %s window [%d, %d] contains frame %d'
                % (hit['Type'], hit['start_frame'], hit['end_frame'], r['restart_frame'])
            )
            expected_type = hit['Type']
        elif r['kind'] == 'kickoff_start':
            status = 'missing_documented'
            evidence = 'kickoff should be inside Kick Off (Start); reconstruction missed it'
            expected_type = 'Kick Off (Start)'
        else:
            # Mid-play restart: schema covers it (SET PIECES exist for throw-in/corner/…),
            # but this notebook has no per-instance tag without the XML.
            status = 'plausible_unverified'
            evidence = (
                'SET PIECES schema covers restarts (%d full-match tags); '
                'no instance-level tag available without XML'
                % n_set_pieces_full
            )
            expected_type = 'Throw-in|Corner Kick|Goal Kick|Free Kick|Kick Off (After Goal)'

        out.append({
            'restart_id': r['restart_id'],
            'restart_frame': r['restart_frame'],
            'kind': r['kind'],
            'restart_to': r['restart_to'],
            'dead_s': r['dead_s'],
            't_from_ko_s': round(r['seconds_from_kickoff'], 1),
            'coverage_status': status,
            'expected_Type': expected_type,
            'evidence': evidence,
        })
    return pd.DataFrame(out)


gt_restart_events = gt_restarts_public(restarts)
coverage_audit = audit_restart_coverage(
    gt_restart_events, documented_kickoffs, SET_PIECE_TYPES)

print('GT restarts in public window:', len(gt_restart_events))
display(gt_restart_events)
print()
print('Coverage audit (no XML):')
display(coverage_audit)

counts = coverage_audit['coverage_status'].value_counts()
n_confirmed = int(counts.get('confirmed', 0))
n_unverified = int(counts.get('plausible_unverified', 0))
n_missing = int(counts.get('missing_documented', 0))
n_total = len(coverage_audit)

print()
print('=== verdict ===')
print('confirmed by documented Smart Tagging facts: %d / %d' % (n_confirmed, n_total))
print('plausible but unverified without XML:         %d / %d' % (n_unverified, n_total))
print('documented coverage failed:                   %d / %d' % (n_missing, n_total))

if n_missing:
    print('FAIL — at least one kickoff is outside the documented Kick Off (Start) window.')
elif n_unverified:
    print(
        'PARTIAL — every kickoff we can check is covered; '
        '%d mid-play restart(s) cannot be confirmed without the Smart Tagging file.'
        % n_unverified
    )
else:
    print('PASS — all listed GT restarts are confirmed covered by documented tags.')

# Budget check: are there enough full-match SET PIECES for the public-window restart count?
n_midplay = int((gt_restart_events['kind'] == 'dead_to_alive').sum())
n_non_ko_tags = int(
    SET_PIECE_TYPES.loc[
        ~SET_PIECE_TYPES['Type'].isin(['Kick Off (Start)']), 'n_full_match'
    ].sum()
)
print()
print('budget (full match, from DATA.md):')
print('  mid-play GT restarts in public 5 min: %d' % n_midplay)
print('  non-Kick-Off-(Start) SET PIECES tags:  %d' % n_non_ko_tags)
print(
    '  enough tags exist globally for these mid-play restarts: %s'
    % ('yes' if n_non_ko_tags >= n_midplay else 'no — tag budget smaller than public restarts')
)

In [ ]:
# Visual summary of the coverage audit on the public timeline.
STATUS_COLOR = {
    'confirmed': '#2A9D8F',
    'plausible_unverified': '#E9C46A',
    'missing_documented': '#E76F51',
}

fig, ax = plt.subplots(figsize=(14, 2.8))
for _, sp in spells.iterrows():
    if sp['label'] == 'X':
        continue
    ax.barh(0, sp['n_frames'], left=sp['start_frame'], height=0.7,
            align='center', color=COLOR[sp['label']], linewidth=0, alpha=0.85)

for _, row in coverage_audit.iterrows():
    ax.axvline(row['restart_frame'], color=STATUS_COLOR[row['coverage_status']],
               lw=2.0, ymin=0.05, ymax=0.95)
    ax.scatter([row['restart_frame']], [0.95],
               color=STATUS_COLOR[row['coverage_status']], s=60, zorder=5,
               edgecolors='black', linewidths=0.5)
    ax.annotate(
        '%s\n%s' % (row['restart_id'], row['coverage_status'].split('_')[0]),
        xy=(row['restart_frame'], 0.95),
        xytext=(0, 8), textcoords='offset points',
        ha='center', va='bottom', fontsize=8,
        color=STATUS_COLOR[row['coverage_status']])

# Documented Kick Off (Start) window on a thin lane.
for _, ko in documented_kickoffs.iterrows():
    if ko['end_frame'] < PUBLIC_FIRST or ko['start_frame'] > PUBLIC_LAST:
        continue
    ax.barh(-0.55, ko['end_frame'] - ko['start_frame'] + 1, left=ko['start_frame'],
            height=0.25, align='center',
            color=SETPIECE_COLOR['Kick Off (Start)'], edgecolor='black', lw=0.5)

ax.set_xlim(PUBLIC_FIRST, PUBLIC_LAST)
ax.set_ylim(-0.85, 1.35)
ax.set_yticks([0])
ax.set_yticklabels(['GT'])
ax.set_xlabel('video frame')
ax.set_title('GT restart coverage by documented Smart Tagging (no XML)')
ax.legend(handles=[
    Patch(facecolor=COLOR['H'], label='H'),
    Patch(facecolor=COLOR['A'], label='A'),
    Patch(facecolor=COLOR['D'], label='D'),
    Line2D([0], [0], color=STATUS_COLOR['confirmed'], lw=2, label='confirmed'),
    Line2D([0], [0], color=STATUS_COLOR['plausible_unverified'], lw=2,
           label='plausible, unverified'),
    Patch(facecolor=SETPIECE_COLOR['Kick Off (Start)'], edgecolor='black',
          label='documented Kick Off (Start)'),
], loc='upper right', ncol=3, fontsize=8, framealpha=0.92)
plt.tight_layout()
plt.show()

print('\ncoverage_audit ready as a DataFrame (%d rows).' % len(coverage_audit))